In [ ]:
import torch
import occupycy
import matplotlib.pyplot as plt

# -- Choose your device --
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
# device = torch.device("cpu")
print(f"Using device: {device}")

seed = 1
torch.manual_seed(seed);

In [ ]:
# 1. Simulate data (arbitrary non-linear functions)
def true_f_psi(X_psi): # here X_psi of shape (n_sites, n_cov_psi=2)
    return torch.sigmoid(
        - 0.5 * torch.cos(X_psi[:,0]) - 0.5 * (X_psi[:,1] ** 2)
    ).unsqueeze(1)
def true_f_d(X_d): # here X_d of shape (n_total_visits, n_cov_d=1)
    return torch.sigmoid(
        torch.tanh(5 * X_d[:,0]) - torch.cos(X_d[:,0])
    ).unsqueeze(1)

y, visit_site_ids, site_ids_in_X_psi, X_psi, X_d, true_z = \
    occupycy.data.simulate_non_linear(
        n_sites=5000,
        n_total_visits_max=int(1e4),
        n_cov_psi=2,
        n_cov_d=1,
        f_effort = lambda x: torch.distributions.LogNormal(0, 3).sample((x.shape[0],)),
        f_psi=true_f_psi,
        f_d=true_f_d,
        seed=seed,
        plot_summary=True,
        verbose=True
    )

In [ ]:
# 2. Build OccuData
# y: tensor of shape (n_total_visits, n_species) with the detection/non-detection data
# visit_site_ids: list of length (n_total_visits,) with the site id of each visit
# site_ids_in_X_psi: list of length (n_sites,) with the site ids of the rows in X_psi
# X_psi: tensor of shape (n_sites, n_cov_psi) with the covariates for occupancy
# X_d: tensor of shape (n_total_visits, n_cov_d) with the covariates for detection
data = occupycy.data.OccuData(
    y=y, 
    visit_site_ids=visit_site_ids,
    X_psi=X_psi,
    X_d=X_d,
    site_ids_in_X_psi=site_ids_in_X_psi
)
data.to(device=device)

# 3. Define a model (MLP)
model = occupycy.models.MLPOccuModel(
    n_species=1,
    n_cov_psi=data.X_psi.shape[1],
    n_cov_d=data.X_d.shape[1],
    hidden_dims_psi=[256, 256], hidden_dims_d=[256, 256],
    activation_fn_psi=torch.nn.ReLU, activation_fn_d=torch.nn.ReLU
)
model.to(device=device)

# 4. Fit the model (Adam)
res = occupycy.fit.fit_sgd_like(
    data, # OccuData object
    model, # OccuModel object
    n_epochs=1000, # maximum number of epochs
    val_fraction=0.15, # fraction of sites to use for validation
    batch_size=512, # number of sites per batch
    optimizer_psi_cls=torch.optim.Adam, # optimizer class for the occupancy part
    optimizer_d_cls=torch.optim.Adam, # optimizer class for the detection part
    lr_psi=1e-3, # learning rate for the occupancy part
    lr_d=1e-3, # learning rate for the detection part
    patience_early_stopping=100,
    patience_reduceLROnPlateau=20,
    # penalization_type='l2_params',
    # penalization_lambda_psi=1e-2,
    # penalization_lambda_d=1e-2,
    device=device
)

# # 3. Define a model (linear)
# model = occupycy.models.LinearOccuModel(
#     n_species=1,
#     n_cov_psi=data.X_psi.shape[1],
#     n_cov_d=data.X_d.shape[1]
# )
# model.to(device=device)

# # 4. Fit the model (L-BFGS full batch)
# res = occupycy.fit.fit_lbfgs_full_batch(
#     data, # OccuData object
#     model, # OccuModel object
#     max_iter=1000, # maximum number of epochs
#     lr=0.1, # fraction of sites to use for validation
#     device=device
# )

In [ ]:
# 5. Get the predictions
with torch.no_grad():
    pred_logit_psi, pred_logit_d = model.forward_logits(X_psi=X_psi.to(device), X_d=X_d.to(device))
    pred_logit_psi, pred_logit_d = pred_logit_psi.cpu(), pred_logit_d.cpu()

In [ ]:
skip_n_first_epochs = 0
n_last_epochs = None

In [ ]:
# PLOT THE LOSSES AND LEARNING RATES

# Extract training history and plot
train_losses = res.history['train_loss']
val_losses = res.history.get('val_loss', [])
lrs_psi = res.history.get('lr_psi', [None] * len(train_losses))
lrs_d = res.history.get('lr_d', [None] * len(train_losses))
epochs = list(range(1, len(train_losses) + 1))
if skip_n_first_epochs > 0:
    train_losses = train_losses[skip_n_first_epochs:]
    val_losses = val_losses[skip_n_first_epochs:]
    lrs_psi = lrs_psi[skip_n_first_epochs:]
    lrs_d = lrs_d[skip_n_first_epochs:]
    epochs = epochs[skip_n_first_epochs:]
if n_last_epochs is not None:
    train_losses = train_losses[-n_last_epochs:]
    val_losses = val_losses[-n_last_epochs:]
    lrs_psi = lrs_psi[-n_last_epochs:]
    lrs_d = lrs_d[-n_last_epochs:]
    epochs = epochs[-n_last_epochs:]

fig, (ax1, ax3) = plt.subplots(
    2, 1, figsize=(6, 4), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]}
)
# Plot losses
ax2 = ax1.twinx()
l1, = ax1.plot(epochs, train_losses, label='Train Loss', color='tab:blue')
if len(val_losses) > 0:
    l2, = ax2.plot(epochs, val_losses, label='Validation Loss', color='tab:orange')
ax1.set_ylabel('Train Loss')
ax2.set_ylabel('Validation Loss')
ax1.set_yscale('log')
ax2.set_yscale('log')
lines = [l1, l2] if len(val_losses) > 0 else [l1]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')
ax1.set_title('Loss histories')
# Plot learning rates
ax3.plot(epochs, lrs_psi, label='lr_psi', color='tab:green')
ax3.plot(epochs, lrs_d, label='lr_d', color='tab:red')
ax3.set_yscale('log')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Learning rate')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# PLOT TRUE VS PREDICTED VALUES

model.to(device='cpu')
true_psi_values = true_f_psi(X_psi).numpy().flatten()
true_d_values = true_f_d(X_d).numpy().flatten()
pred_psi_values = torch.sigmoid(pred_logit_psi).detach().numpy().flatten()
pred_d_values = torch.sigmoid(pred_logit_d).detach().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

def plot_identity_line(ax, x, y):
    min_val = max(x.min(), y.min())
    max_val = min(x.max(), y.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7)

# Occupancy
axes[0].scatter(
    true_psi_values, pred_psi_values, alpha=0.05, s=15, 
    color='tab:blue', edgecolor='k', linewidths=0.3
)
plot_identity_line(axes[0], true_psi_values, pred_psi_values)
axes[0].set_xlabel('True Occupancy Probability (ψ)')
axes[0].set_ylabel('Predicted Occupancy Probability (ψ)')
axes[0].set_title('Occupancy: True vs Predicted')
# Detection
axes[1].scatter(
    true_d_values, pred_d_values, alpha=0.002, s=0.3, 
    color='tab:orange'
)
plot_identity_line(axes[1], true_d_values, pred_d_values)
axes[1].set_xlabel('True Detection Probability (d)')
axes[1].set_ylabel('Predicted Detection Probability (d)')
axes[1].set_title('Detection: True vs Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Utils plots

def plot_occupancy(ax, X_psi, f_psi, vmin=None, vmax=None, cmap='viridis', 
                   col_label='Occupancy Probability ($\\psi$)'):
    values = f_psi(X_psi).numpy().flatten()
    # Background: True surface (contour)
    x1_grid, x2_grid = torch.meshgrid(
        torch.linspace(X_psi[:,0].min(), X_psi[:,0].max(), 100),
        torch.linspace(X_psi[:,1].min(), X_psi[:,1].max(), 100),
        indexing='ij'
    )
    X_psi_grid = torch.stack([x1_grid.flatten(), x2_grid.flatten()], dim=1)
    psi_grid = f_psi(X_psi_grid).reshape(100, 100).detach().numpy()
    if vmin is None: 
        vmin = psi_grid.min()
    if vmax is None:
        vmax = psi_grid.max()
    contour = ax.contourf(
        x1_grid, x2_grid, psi_grid, levels=50, cmap=cmap, alpha=0.1, 
        vmin=vmin, vmax=vmax
    )
    # Overlay: Actual site points colored by psi
    sc = ax.scatter(
        X_psi[:,0], X_psi[:,1], c=values, cmap=cmap, 
        s=15, edgecolor='k', linewidths=0.1, alpha=0.2,
        vmin=vmin, vmax=vmax
    )
    plt.colorbar(sc, label=col_label, ax=ax)
    ax.set_xlabel('$X_{\\psi, 0}$')
    ax.set_ylabel('$X_{\\psi, 1}$')

def plot_detection(ax, X_d, f_d, label, color):
    values = f_d(X_d).numpy().flatten()
    # True function as line
    x_d_grid = torch.linspace(X_d[:,0].min(), X_d[:,0].max(), 200)
    X_d_grid = x_d_grid.unsqueeze(1)
    d_grid = f_d(X_d_grid).flatten().detach().numpy()
    ax.plot(x_d_grid, d_grid, color=color, label=label)
    ax.scatter(
        X_d[:,0], values, alpha=0.3, s=15, color=color,
        edgecolor='k', linewidths=0.3,
    )
    ax.set_xlabel('$X_{d, 0}$')
    ax.set_ylabel('Detection Probability ($d$)')

In [ ]:
# PLOT TRUE AND PREDICTED OCCUPANCY AND DETECTION FUNCTIONS

fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(3, 3, height_ratios=[4, 3, 1])
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])
ax02 = fig.add_subplot(gs[0, 2])
ax10 = fig.add_subplot(gs[1, :])
ax20 = fig.add_subplot(gs[2, :])

def f_pred_psi(X_psi):
    with torch.no_grad():
        pred_logit_psi = model.predict_logit_psi(X_psi)
        return torch.sigmoid(pred_logit_psi)
def f_pred_d(X_d):
    with torch.no_grad():
        pred_logit_d = model.predict_logit_d(X_d)
        return torch.sigmoid(pred_logit_d)

vmin_psi = min(true_psi_values.min(), pred_psi_values.min())
vmax_psi = max(true_psi_values.max(), pred_psi_values.max())

# Occupancy (true)
plot_occupancy(ax00, X_psi, true_f_psi, vmin=vmin_psi, vmax=vmax_psi)
ax00.set_title('TRUE Occupancy ($\\psi$)')
# Occupancy (predicted)
plot_occupancy(ax01, X_psi, f_pred_psi, vmin=vmin_psi, vmax=vmax_psi)
ax01.set_title('PREDICTED Occupancy ($\\psi$)')
# Occupancy (predicted-true)
f_rel_error = lambda x: ((f_pred_psi(x) - true_f_psi(x)) / true_f_psi(x)).abs()
# vmin_rel_err = max(f_rel_error(X_psi).numpy().flatten().min(), -0.5)
vmin_rel_err = 0
vmax_rel_err = min(f_rel_error(X_psi).numpy().flatten().max(), 1)
plot_occupancy(
    ax02, X_psi, f_rel_error, 
    cmap='coolwarm', 
    vmin=vmin_rel_err, vmax=vmax_rel_err,
    col_label='Relative Error'
)
ax02.set_title('Relative Error in Occupancy')
# Detection
plot_detection(ax10, X_d, true_f_d, label='True Detection', color='seagreen')
plot_detection(ax10, X_d, f_pred_d, label='Predicted Detection', color='royalblue')
ax10.legend()
ax10.set_title('Detection ($d$)')
# Histogram of x_d values
ax20.hist(X_d[:,0].numpy(), bins=30, color='gray', edgecolor='k', alpha=0.7)
ax20.set_xlabel('$X_{d, 0}$')
ax20.set_ylabel('Frequency')
ax20.invert_yaxis()
for sp in ['bottom', 'left', 'right']:
    ax20.spines[sp].set_visible(False)
ax20.xaxis.set_ticks_position('top')
ax20.yaxis.set_ticks([])
# ax20.set_xlabel('')
ax20.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# COMPUTE AUC

ground_truth = torch.tensor(data.y.cpu().numpy().flatten(), dtype=torch.float32)
model_scores = torch.tensor(
    pred_psi_values[visit_site_ids] * pred_d_values, 
    dtype=torch.float32
)
true_scores = torch.tensor(
    true_psi_values[visit_site_ids] * true_d_values,
    dtype=torch.float32
)

def auc_torch(y, scores):
    y = y.bool()
    scores = scores

    pos = scores[y]
    neg = scores[~y]

    # all pairwise comparisons
    diff = pos[:, None] - neg[None, :]

    return (diff > 0).float().mean() + 0.5 * (diff == 0).float().mean()

auc_model = auc_torch(ground_truth, model_scores)
auc_true = auc_torch(ground_truth, true_scores)
print('-- AUC ROC on PO data --')
print(f'Model: {auc_model.item():.3f}')
print(f'True: {auc_true.item():.3f}')

# Expected:
# Linear model: 0.646 (LBFGS - cpu, cuda, mps)
# MLP model:    0.685 (Adam - cpu, cuda, mps)
# True model:   0.662